# Beat detector, heart rate and pauses on MIT-BIH Arrhythmia Database (Notebook 6)
**Bektur Asilov · COMSE-23 · Ala-Too**

Every number the system shows — AF decisions, heart rate, slow/fast-rate events, pauses — starts with our R-peak
detector (the same algorithm in the firmware, server and app). Here it is checked against the cardiologist beat
annotations of all 48 MIT-BIH Arrhythmia records (30 min each, 360 Hz, MLII lead), which the detector was never tuned on.

Rules fixed before the run (standard practice, ANSI/AAMI EC57):
* a detected beat matches a reference beat within **150 ms**, with no delay correction; the full 30 min are scored
  (EC57 allows skipping the first 5 min as warm-up; we do not, which is stricter);
* **heart rate** = beats in each 10-s window × 6, compared with the reference count (mean absolute error, bpm);
* **slow / fast rate**: 10-s windows with reference rate < 40 or > 150 bpm — how many we also flag;
* **pauses**: reference RR ≥ 3 s — how many we also report.

> Kaggle: Internet **On** (MITDB ≈ 100 MB from the PhysioNet open-data mirror, fallback physionet.org). CPU is enough.

In [ ]:
import os, time, json, subprocess, sys
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
DB = os.environ.get('MITDB_DIR')
RECS = [100,101,102,103,104,105,106,107,108,109,111,112,113,114,115,116,117,118,119,121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,219,220,221,222,223,228,230,231,232,233,234]
if not DB:
    DB = '/kaggle/tmp/mitdb'; os.makedirs(DB, exist_ok=True)
    for r in RECS:
        for ext in ('hea', 'dat', 'atr'):
            for base in ('https://physionet-open.s3.amazonaws.com/mitdb/1.0.0/', 'https://physionet.org/files/mitdb/1.0.0/'):
                if subprocess.run(['curl', '-fsSL', '--max-time', '300', '-o', f'{DB}/{r}.{ext}', base + f'{r}.{ext}']).returncode == 0: break
            else:
                raise RuntimeError(f'download failed: {r}.{ext}')
print('records:', len(RECS))

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

BEAT = set('NLRBAaJSVrFejnE/fQ?')

In [ ]:
def match(ref, det, tol):
    j = np.searchsorted(det, ref); used = np.zeros(len(det), bool); tp = 0
    for r, i in zip(ref, j):
        best = None
        for k in (i - 1, i):
            if 0 <= k < len(det) and not used[k] and abs(det[k] - r) <= tol and (best is None or abs(det[k] - r) < abs(det[best] - r)):
                best = k
        if best is not None: used[best] = True; tp += 1
    return tp

rows, hr_rows, T0 = [], [], time.time()
for r in RECS:
    rec = wfdb.rdrecord(f'{DB}/{r}', channels=[0]); ann = wfdb.rdann(f'{DB}/{r}', 'atr')
    fs = rec.fs; x = to_125(rec.p_signal[:, 0], fs); n_s = len(x) / FS
    ref = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT]) * FS / fs
    det = detect_r(x).astype(float)
    tp = match(ref, det, 0.15 * FS)
    near = det[np.clip(np.searchsorted(det, ref), 0, len(det) - 1)] if len(det) else np.array([])
    rows.append(dict(record=r, ref_beats=len(ref), det_beats=len(det), TP=tp, FN=len(ref) - tp, FP=len(det) - tp,
                     median_offset_ms=float(np.median(near - ref) / FS * 1000) if len(near) else np.nan))
    for w in range(0, int(n_s // 10)):
        a, b = w * 10 * FS, (w + 1) * 10 * FS
        hr_rows.append(dict(record=r, w=w, ref_bpm=((ref >= a) & (ref < b)).sum() * 6, det_bpm=((det >= a) & (det < b)).sum() * 6))
    rr_ref = np.diff(ref) / FS; rr_det = np.diff(det) / FS
    rows[-1].update(ref_pauses=int((rr_ref >= 3).sum()), det_pauses=int((rr_det >= 3).sum()),
                    det_pauses_matching=int(sum(any(abs(d0 - r0) < 2 * FS for d0 in det[:-1][rr_det >= 3]) for r0 in ref[:-1][rr_ref >= 3])))
B = pd.DataFrame(rows); H = pd.DataFrame(hr_rows)
B.to_csv(f'{OUT_DIR}/mitdb_beats.csv', index=False); H.to_csv(f'{OUT_DIR}/mitdb_hr.csv', index=False)
print(f'{time.time()-T0:.0f} s')

## Results

In [ ]:
tp, fn, fp = B.TP.sum(), B.FN.sum(), B.FP.sum()
H['err'] = (H.det_bpm - H.ref_bpm).abs()
slow, fast = H.ref_bpm < 40, H.ref_bpm > 150
S = {
  'beats_reference': int(B.ref_beats.sum()),
  'beat_sensitivity': tp / (tp + fn), 'beat_ppv': tp / (tp + fp),
  'records_with_sensitivity_below_95pct': B.assign(se=B.TP / B.ref_beats).query('se < 0.95').record.tolist(),
  'hr_mae_bpm_10s': float(H.err.mean()), 'hr_within_5bpm': float((H.err <= 5).mean()),
  'slow_windows_ref': int(slow.sum()), 'slow_windows_also_flagged': int((slow & (H.det_bpm < 40)).sum()),
  'fast_windows_ref': int(fast.sum()), 'fast_windows_also_flagged': int((fast & (H.det_bpm > 150)).sum()),
  'false_slow_windows': int((~slow & (H.det_bpm < 40)).sum()), 'false_fast_windows': int((~fast & (H.det_bpm > 150)).sum()),
  'windows': int(len(H)), 'median_detection_offset_ms': float(B.median_offset_ms.median()),
  'pauses_ref': int(B.ref_pauses.sum()), 'pauses_found': int(B.det_pauses_matching.sum()), 'pauses_reported_by_us': int(B.det_pauses.sum()),
}
json.dump(S, open(f'{OUT_DIR}/mitdb_summary.json', 'w'), indent=2, default=float)
print(json.dumps(S, indent=2, default=lambda v: round(float(v), 4)))
pd.set_option('display.width', 200); print(B.assign(se=(B.TP / B.ref_beats).round(4), ppv=(B.TP / B.det_beats).round(4)).to_string(index=False))